# Solar Filament Segmentation Challenge 2026: end-to-end pipeline

This notebook runs the whole method from raw GONG H-alpha JPEGs to `submission.csv`:

1. **Preprocessing.** Fit the solar disk at the steepest limb edge, divide out limb darkening (median in thin rings), and add a radial-distance channel.
2. **Pixels.** A ResNet34 U-Net trained on native-resolution 512 px crops against the mean of the annotators' masks. Inference uses overlapping 1024 px tiles with Hann blending and 8 flips/rotations.
3. **Instances.** A YOLO11-seg detector, trained with each annotator record as its own sample, decides which U-Net pixels form one filament. Detections claim pixels in order of confidence, so filaments never overlap.
4. **Post-processing**, tuned for Panoptic Quality on a held-out fold with an exact copy of the organizers' scoring code.

**Modes**
- `MODE = "train"`: prepare the data, train both models, validate and tune on fold `FOLD`, then predict the test set. Takes about 3–4 h on Kaggle with GPU T4×2 (the two models train in parallel).
- `MODE = "reproduce"`: load the committed weights from `weights/` and predict the test set only. No training, about 15 min.

**Kaggle settings:** Accelerator *GPU T4×2*, Internet *on* (for pip and the ImageNet/COCO backbone weights). Add the competition data. Then either attach this repository as a Kaggle dataset, or set `REPO_URL` below.

In [ ]:
import glob, json, os, shutil, subprocess, sys, time

REPO_URL = "https://github.com/YOUR-USERNAME/solar-filament-segmentation.git"  # <- your public repository
ON_KAGGLE = os.path.isdir("/kaggle/working")
REPO_DIR = "/kaggle/working/solar-filament" if ON_KAGGLE else os.path.abspath("..")

if ON_KAGGLE and not os.path.isdir(REPO_DIR):
    attached = glob.glob("/kaggle/input/**/src/cli.py", recursive=True)
    if attached:  # the repository uploaded as a Kaggle dataset
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO_DIR)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
print("working in", os.getcwd())

In [ ]:
import torch
from src.config import load_config

MODE = "train"            # "train" or "reproduce"
FOLD = 0                  # held-out fold for validation and tuning; -1 = train on all data (no validation)
PREDICT_FOLDS = [FOLD]    # model folds ensembled for the test predictions, e.g. [0, 1, 2, 3, 4]
SUBMISSION = "/kaggle/working/submission.csv" if ON_KAGGLE else "outputs/submission.csv"
OVERRIDES = []            # config overrides, e.g. ["unet.epochs=20", "postproc.min_area=150"]

cfg = load_config(overrides=OVERRIDES)
os.makedirs("outputs", exist_ok=True)
print("GPUs:", torch.cuda.device_count(), "| mode:", MODE, "| fold:", FOLD)

## 1. Preprocess the training data and build the folds

In [ ]:
if MODE == "train":
    from src.pipeline import prepare
    cache = prepare(cfg)
    import pandas as pd
    folds = pd.read_csv(cache / "folds.csv")
    print(folds.groupby("fold")[["image_id", "n_records"]].agg({"image_id": "count", "n_records": "sum"}))

## 2. Train the U-Net and the detector (in parallel when two GPUs are available)

In [ ]:
def run_jobs(jobs: dict, parallel: bool) -> None:
    """Runs CLI jobs as subprocesses (one GPU each when parallel) and prints their progress."""
    def start(i, args, log):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i if parallel else 0))
        cmd = [sys.executable, "-m", "src.cli", *args] + [x for o in OVERRIDES for x in ("--set", o)]
        return subprocess.Popen(cmd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

    def last_line(log):
        text = open(log, errors="ignore").read().replace("\r", "\n").strip().splitlines()
        return text[-1][:160] if text else ""

    queue = list(jobs.items())
    running = {}
    while queue or running:
        while queue and (parallel or not running):
            name, (args, log) = queue.pop(0)
            running[name] = (start(len(running), args, log), log)
        time.sleep(60)
        for name, (proc, log) in list(running.items()):
            print(f"[{time.strftime('%H:%M')}] {name}: {last_line(log)}", flush=True)
            if proc.poll() is not None:
                if proc.returncode != 0:
                    raise RuntimeError(f"{name} failed; see {log}")
                del running[name]

if MODE == "train":
    run_jobs({"unet": (["train-unet", "--fold", str(FOLD)], "outputs/train_unet.log"),
              "yolo": (["train-yolo", "--fold", str(FOLD)], "outputs/train_yolo.log")},
             parallel=torch.cuda.device_count() >= 2)
    print(sorted(os.listdir(cfg["weights_dir"])))

## 3. Validate on the held-out fold and tune the post-processing

Scoring follows the organizers' self-evaluation notebook: every annotator record is scored separately, a match needs IoU > 0.5, and TP/FP/FN are pooled. The tuned parameters are written to `weights/postproc.json`, which `predict` uses.

In [ ]:
if MODE == "train" and FOLD >= 0:
    import matplotlib.pyplot as plt
    import pandas as pd
    from src.eval.pq import evaluate
    from src.eval.plots import plot_diagnostics
    from src.pipeline import validate

    report = validate(cfg, FOLD)
    print(json.dumps(report, indent=2))
    read = lambda p: pd.read_csv(p, dtype=str, keep_default_na=False)
    result = evaluate(read(f"outputs/gt_fold{FOLD}.csv"), read(f"outputs/oof_fold{FOLD}.csv"))
    plot_diagnostics(result, f"Validation fold {FOLD}").savefig(f"outputs/diagnostics_fold{FOLD}.png", dpi=120)
    plt.show()

## 4. Predict the test set and write `submission.csv`

In [ ]:
from src.pipeline import predict

predict(cfg, PREDICT_FOLDS, SUBMISSION)

## 5. Look at the predictions

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from src.data.paths import find_data, list_images
from src.data.preprocess import read_gray
from src.eval.plots import overlay
from src.submit.rle import decode_rle

sub = pd.read_csv(SUBMISSION, dtype=str, keep_default_na=False)
images = list_images(find_data(cfg["data"]["root"]).test_images)
stems = sub["filament_id"].str.split("_", n=1).str[0]
for image_id in list(images)[:3]:
    masks = [decode_rle(r) for r in sub.loc[stems == image_id, "segmentation_rle"]]
    plt.figure(figsize=(11, 11))
    plt.imshow(overlay(read_gray(images[image_id]), masks))
    plt.title(f"{image_id}: {len(masks)} filaments")
    plt.axis("off")
    plt.show()

## 6. Record the exact package versions

Paste this output into `requirements.txt`, so the judges get the environment that produced the submission.

In [ ]:
import importlib.metadata as md

for name in ["numpy", "pandas", "scipy", "opencv-python", "opencv-python-headless", "pycocotools", "PyYAML",
             "matplotlib", "torch", "torchvision", "timm", "segmentation-models-pytorch", "ultralytics"]:
    try:
        print(f"{name}=={md.version(name)}")
    except md.PackageNotFoundError:
        pass

## After a training run

1. Download `weights/` from the notebook output: `unet_resnet34_fold*.pt`, `yolo11m_seg_1536_fold*.pt` and `postproc.json`.
2. Commit them to the repository. Each file is under 100 MB, so plain git works without LFS.
3. Submit `/kaggle/working/submission.csv` to the competition.

With the weights committed, `MODE = "reproduce"` regenerates the same submission without retraining, which is what the judges will run.